In [2]:
import pandas as pd

Load your data:

In [3]:
df = pd.read_csv("D:/Projects/chicago-bike-demand-intelligence/02_data_processed/city_hourly_demand_weather.csv")

df.head()

,date,hour,trip_count,member_trip_count,casual_trip_count,avg_trip_duration_min,is_weekend,rush_hour,temperature,precipitation_mm,wind_speed
0,2025-03-01,0,122,69,53,8.53,True,False,1.9,0.0,18.3
1,2025-03-01,1,85,52,33,8.42,True,False,0.8,0.0,17.2
2,2025-03-01,2,46,26,20,7.60,True,False,-0.2,0.0,22.1
3,2025-03-01,3,31,17,14,6.50,True,False,-1.6,0.0,19.8
4,2025-03-01,4,20,14,6,8.61,True,False,-2.1,0.0,20.2


Create Weather Features

In [4]:
df["is_raining"] = df["precipitation_mm"] > 0

Temperature buckets

In [5]:
def temp_bucket(temp):
    if temp < 5:
        return "cold"
    elif temp < 20:
        return "mild"
    else:
        return "warm"

df["temp_bucket"] = df["temperature"].apply(temp_bucket)

Bad weather flag

In [6]:
df["bad_weather"] = (
    (df["precipitation_mm"] > 0) |
    (df["temperature"] < 5)
)

Sanity check

In [7]:
print(df[["temperature", "temp_bucket"]].head())
print(df["is_raining"].value_counts())
print(df["temp_bucket"].value_counts())

   temperature temp_bucket
0          1.9        cold
1          0.8        cold
2         -0.2        cold
3         -1.6        cold
4         -2.1        cold
is_raining
False    1913
True      294
Name: count, dtype: int64
temp_bucket
mild    1513
cold     546
warm     148
Name: count, dtype: int64


Rain vs demand

In [8]:
df.groupby("is_raining")["trip_count"].mean()

is_raining
False    553.018819
True     377.442177
Name: trip_count, dtype: float64

Temperature vs demand

In [9]:
df.groupby("temp_bucket")["trip_count"].mean()

temp_bucket
cold     289.084249
mild     556.329808
warm    1144.094595
Name: trip_count, dtype: float64

Members vs casual

In [10]:
df.groupby("temp_bucket")[["member_trip_count", "casual_trip_count"]].mean()

,member_trip_count,casual_trip_count
temp_bucket,,
cold,227.119048,61.965201
mild,372.337740,183.992069
warm,715.763514,428.331081


### Rain Impact on Demand

Demand decreases significantly during rainy conditions.

- No rain: ~553 avg trips
- Rain: ~377 avg trips

This represents roughly a 30% drop in demand, indicating strong sensitivity to precipitation.

### Temperature Impact

Demand increases substantially with temperature:

- Cold: ~289 trips
- Mild: ~556 trips
- Warm: ~1144 trips

Warm conditions generate nearly 4x the demand compared to cold weather.

### Rider Behavior by Weather

Casual riders are significantly more affected by weather than members.

- Casual usage increases dramatically in warm conditions
- Member usage remains more stable across conditions

This suggests casual riders are more discretionary, while members rely on bikes for routine transportation.

### Event Data Integration

Load events into your notebook

In [11]:
events = pd.read_csv("D:/Projects/chicago-bike-demand-intelligence/02_data_processed/events.csv")

events["event_date"] = pd.to_datetime(events["event_date"]).dt.date

events.head()

,event_name,event_date,event_type
0,Chicago Cubs Home Game,2025-03-18,sports
1,Chicago Cubs Home Game,2025-03-19,sports
2,Chicago Cubs Home Game,2025-04-08,sports
3,Chicago Cubs Home Game,2025-04-09,sports
4,Chicago Cubs Home Game,2025-05-28,sports


Merge with your main dataset

In [12]:
df["date"] = pd.to_datetime(df["date"]).dt.date

In [13]:
df = df.merge(
    events,
    left_on="date",
    right_on="event_date",
    how="left"
)

Create event flag

In [14]:
df["is_event_day"] = df["event_name"].notna()

Clean up

In [15]:
df = df.drop(columns=["event_date"])

Validate

In [16]:
df["is_event_day"].value_counts()

is_event_day
False    1919
True      288
Name: count, dtype: int64

Basic impact

In [17]:
df.groupby("is_event_day")["trip_count"].mean()

is_event_day
False    534.150599
True     499.506944
Name: trip_count, dtype: float64

Event + weather

In [18]:
df.groupby(["is_event_day", "is_raining"])["trip_count"].mean()

is_event_day  is_raining
False         False         560.969369
              True          358.350394
True          False         499.641129
              True          498.675000
Name: trip_count, dtype: float64

Event + temperature

In [19]:
df.groupby(["is_event_day", "temp_bucket"])["trip_count"].mean()

is_event_day  temp_bucket
False         cold            283.025918
              mild            562.566138
              warm           1125.706767
True          cold            322.879518
              mild            512.905263
              warm           1307.133333
Name: trip_count, dtype: float64

Member vs casual on event days

In [20]:
df.groupby("is_event_day")[["member_trip_count", "casual_trip_count"]].mean()

,member_trip_count,casual_trip_count
is_event_day,,
False,359.553934,174.596665
True,358.690972,140.815972


Final Dataset

In [21]:
df.to_csv("D:/Projects/chicago-bike-demand-intelligence/02_data_processed/final_demand_dataset.csv", index=False)

In [22]:
print(df.columns.tolist())

['date', 'hour', 'trip_count', 'member_trip_count', 'casual_trip_count', 'avg_trip_duration_min', 'is_weekend', 'rush_hour', 'temperature', 'precipitation_mm', 'wind_speed', 'is_raining', 'temp_bucket', 'bad_weather', 'event_name', 'event_type', 'is_event_day']


In [54]:
df = df[
    [
        "date",
        "hour",
        "trip_count",
        "member_trip_count",
        "casual_trip_count",
        "avg_trip_duration_min",
        "is_weekend",
        "rush_hour",
        "temperature",
        "precipitation_mm",
        "wind_speed",
        "is_raining",
        "temp_bucket",
        "bad_weather",
        "is_event_day",
    ]
]

In [23]:
print(df.columns.tolist())
print(df.head())
print(df.shape)

['date', 'hour', 'trip_count', 'member_trip_count', 'casual_trip_count', 'avg_trip_duration_min', 'is_weekend', 'rush_hour', 'temperature', 'precipitation_mm', 'wind_speed', 'is_raining', 'temp_bucket', 'bad_weather', 'event_name', 'event_type', 'is_event_day']
         date  hour  trip_count  member_trip_count  casual_trip_count  \
0  2025-03-01     0         122                 69                 53   
1  2025-03-01     1          85                 52                 33   
2  2025-03-01     2          46                 26                 20   
3  2025-03-01     3          31                 17                 14   
4  2025-03-01     4          20                 14                  6   

   avg_trip_duration_min  is_weekend  rush_hour  temperature  \
0                   8.53        True      False          1.9   
1                   8.42        True      False          0.8   
2                   7.60        True      False         -0.2   
3                   6.50        True      F

In [24]:
df.to_csv(
    "D:/Projects/chicago-bike-demand-intelligence/02_data_processed/final_demand_weather_events.csv",
    index=False
)

In [25]:
df.groupby("is_event_day")[["member_trip_count", "casual_trip_count"]].mean()

,member_trip_count,casual_trip_count
is_event_day,,
False,359.553934,174.596665
True,358.690972,140.815972
